# 06 Deployment: Dashboard

ขั้นตอนนี้สร้าง dashboard 1 หน้าเป็นไฟล์ HTML ที่เปิดได้โดยไม่ต้องต่ออินเทอร์เน็ต ตาม Scope หัวข้อ 18 ฉบับปรับตามผลขั้น 05

**หลักการ**
- **origin ล่าสุด:** ใช้ origin ล่าสุดที่ไม่เกินวันที่ดาวน์โหลดข้อมูล (as-of) เพื่อให้ตรงกับกฎ forecast origin วันที่ 15 ของ t+2
- **โมเดล:** ใช้โมเดลที่เลือกจาก validation ในขั้น 05 พร้อม alpha หรือ ARIMA order ที่ตรึงไว้ แล้ว fit ใหม่ด้วยแถว training ที่ target เผยแพร่แล้ว ณ origin นั้น
- **ช่วงความคลาดเคลื่อน:** ± MAE ของช่วง test ของโมเดลเดียวกันที่ horizon เดียวกัน
- **ความโปร่งใส:** แสดงผลการประเมินจริงคู่กับค่าทำนายทุกครั้ง รวมถึงข้อที่ว่าสัญญาณ High Risk ไม่ได้ดีกว่าการเตือนทุกเดือน
- **ผลรอบที่ 2:** แสดงเป็นตารางและติดป้ายเชิงสำรวจ

In [1]:
from __future__ import annotations
import hashlib, html, json, warnings
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
import plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.arima.model import ARIMA

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'PROJECT_SCOPE.md').exists())
RAW_DIR, PROCESSED_DIR, INTERIM_DIR = ROOT / 'data/raw', ROOT / 'data/processed', ROOT / 'data/interim'
METRICS_DIR, DASH_DIR = ROOT / 'logs/metrics', ROOT / 'reports/dashboard'
DASH_DIR.mkdir(parents=True, exist_ok=True)
GROUPS = ['318', '402']
GROUP_TH = {'318': 'ปุ๋ย และยากำจัดศัตรูพืช (318)', '402': 'นมและผลิตภัณฑ์นม (402)'}
GROUP_COLOR = {'318': '#2a78d6', '402': '#1baf7a'}
INK, MUTED, GRID = '#2b2b2b', '#6b6b66', '#e6e5df'
TH_MONTHS = ['ม.ค.', 'ก.พ.', 'มี.ค.', 'เม.ย.', 'พ.ค.', 'มิ.ย.', 'ก.ค.', 'ส.ค.', 'ก.ย.', 'ต.ค.', 'พ.ย.', 'ธ.ค.']
def th_month(ts): ts = pd.Timestamp(ts); return f'{TH_MONTHS[ts.month - 1]} {ts.year + 543}'
def th_date(ts): ts = pd.Timestamp(ts); return f'{ts.day} {th_month(ts)}'

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''): digest.update(chunk)
    return digest.hexdigest()

manifest = pd.read_csv(METRICS_DIR / '03_processed_manifest.csv')
assert sha256(ROOT / manifest.loc[0, 'file']) == manifest.loc[0, 'sha256']
dates = ['month_ce', 'forecast_origin', 'target_month', 'target_available_date']
modeling = pd.read_csv(ROOT / manifest.loc[0, 'file'], dtype={'commodity_code': str}, parse_dates=dates)
latest_features = pd.read_csv(PROCESSED_DIR / 'latest_features.csv', dtype={'commodity_code': str},
                              parse_dates=['month_ce', 'forecast_origin', 'fx_lead_t1_full_max_date', 'brent_lead_t1_full_max_date'])
FEATURE_SETS = {r.feature_set: r.features.split('|') for r in
                pd.read_csv(METRICS_DIR / '03_feature_sets.csv', dtype={'feature_set': str}).itertuples()}
best = pd.read_csv(METRICS_DIR / '05_best_model_by_validation.csv', dtype={'commodity_code': str})
test_metrics = pd.read_csv(METRICS_DIR / '05_test_metrics.csv', dtype={'commodity_code': str})
rq4 = pd.read_csv(METRICS_DIR / '05_rq4_horizon_skill.csv', dtype={'commodity_code': str})
alerts = pd.read_csv(METRICS_DIR / '05_alert_metrics.csv', dtype={'commodity_code': str})
dm_main = pd.read_csv(METRICS_DIR / '05_dm_main_comparisons.csv', dtype={'commodity_code': str})
iter2 = pd.read_csv(METRICS_DIR / '05b_results.csv', dtype={'commodity_code': str})
passthrough = pd.read_csv(METRICS_DIR / '02_passthrough_summary.csv')
alphas = pd.read_csv(METRICS_DIR / '04_ridge_alpha_selection.csv', dtype={'commodity_code': str, 'feature_set': str})
arima_grid = pd.read_csv(METRICS_DIR / '04_arima_order_selection.csv', dtype={'commodity_code': str})
pred04 = pd.read_csv(METRICS_DIR / '04_predictions.csv', dtype={'commodity_code': str, 'feature_set': str}, parse_dates=['month_ce'])
pred04['series'] = pred04.model + '/' + pred04.feature_set
external = pd.read_csv(INTERIM_DIR / 'external_monthly_aggregates.csv', parse_dates=['month_ce'])

sources = json.loads((RAW_DIR / 'sources.json').read_text(encoding='utf-8'))['sources']
imi_source = next(x for x in reversed(sources) if x['source_id'].startswith('imi-monthly-'))
assert sha256(RAW_DIR / imi_source['file']) == imi_source['sha256']
raw = pd.read_csv(RAW_DIR / imi_source['file'], dtype={'commodityCode': str})
raw['commodityCode'] = raw.commodityCode.str.zfill(3)
raw['month_ce'] = pd.to_datetime(dict(year=raw.year - 543, month=raw.month, day=1))
imi_series = {g: raw[raw.commodityCode == g].set_index('month_ce')['index'].sort_index().asfreq('MS') for g in GROUPS}
AS_OF = pd.Timestamp(imi_source['downloaded_at_utc'][:10])
print(f'as-of (วันที่ดาวน์โหลดข้อมูล): {AS_OF.date()}; plotly {plotly.__version__}')

as-of (วันที่ดาวน์โหลดข้อมูล): 2026-10-05; plotly 7.1.0


## 1. Origin ล่าสุดและฟังก์ชันพยากรณ์

เลือกเดือนฐาน t ล่าสุดที่ forecast origin ไม่เกินวันที่ as-of แล้วตรวจว่าข้อมูลภายนอกเดือน t+1 ครบก่อน origin

ฟังก์ชันพยากรณ์ใช้กฎเดียวกับขั้น 04 จึงตรวจความถูกต้องได้โดยคำนวณค่าทำนายของ origin ในช่วง test ซ้ำ แล้วเทียบกับ `04_predictions.csv` ต้องได้ค่าเท่ากัน

In [2]:
eligible = latest_features[latest_features.forecast_origin <= AS_OF]
T = eligible.month_ce.max()
ORIGIN = T + pd.offsets.MonthBegin(2) + pd.Timedelta(days=14)
latest_rows = eligible[eligible.month_ce == T].set_index('commodity_code')
assert (latest_rows.fx_lead_t1_full_max_date < ORIGIN).all() and (latest_rows.brent_lead_t1_full_max_date < ORIGIN).all()
assert all(imi_series[g].index.max() >= T for g in GROUPS)
print(f'เดือนฐาน t = {T:%Y-%m}; forecast origin = {ORIGIN.date()}; ข้อมูลภายนอกเดือน t+1 ถึง {latest_rows.fx_lead_t1_full_max_date.max().date()}')

BEST = {(r.commodity_code, int(r.horizon)): r.series for r in best.itertuples()}
arima_sel = arima_grid[arima_grid.converged].sort_values('aic').groupby('commodity_code').head(1).set_index('commodity_code')

def train_rows(g, h, origin):
    panel = modeling[(modeling.commodity_code == g) & (modeling.horizon == h)]
    train = panel[panel.target_available_date <= origin]
    assert train.month_ce.max() <= origin - pd.DateOffset(months=h + 2)
    return train

def forecast(g, h, t, features_row, origin):
    series = BEST[(g, h)]
    model, fs = series.split('/')
    train = train_rows(g, h, origin)
    out = {'series': series, 'train_n': len(train), 'threshold': float(np.quantile(train.target_y_pct, 0.8))}
    if model == 'ridge':
        feats = FEATURE_SETS[fs]
        alpha = float(alphas[(alphas.commodity_code == g) & (alphas.horizon == h) & (alphas.feature_set == fs) & alphas.selected].alpha.iloc[0])
        scaler = StandardScaler().fit(train[feats])
        ridge = Ridge(alpha=alpha).fit(scaler.transform(train[feats]), train.target_y_pct)
        x = scaler.transform(features_row[feats].to_frame().T.astype(float))
        out.update(y_pred=float(ridge.predict(x)[0]), alpha=alpha,
                   coefficients=pd.Series(ridge.coef_, index=feats))
    elif model == 'arima':
        sel = arima_sel.loc[g]
        log_imi = np.log(imi_series[g][:t])
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            result = ARIMA(log_imi, order=(int(sel.p), 1, int(sel.q)), trend=sel.trend).fit()
        out.update(y_pred=float(100 * (np.asarray(result.forecast(steps=h))[-1] - log_imi.iloc[-1])),
                   order=f'ARIMA({int(sel.p)},1,{int(sel.q)})')
    else:
        raise ValueError(series)
    return out

# ตรวจ: คำนวณค่าทำนายของ origin สุดท้ายในช่วง test ซ้ำ แล้วต้องเท่ากับผลขั้น 04
checks = []
for g in GROUPS:
    for h in range(2, 7):
        row = modeling[(modeling.commodity_code == g) & (modeling.horizon == h)].sort_values('month_ce').iloc[-1]
        again = forecast(g, h, row.month_ce, row, row.forecast_origin)['y_pred']
        stored = pred04[(pred04.commodity_code == g) & (pred04.horizon == h) & (pred04.series == BEST[(g, h)])
                        & (pred04.month_ce == row.month_ce)].y_pred.iloc[0]
        checks.append({'group': g, 'h': h, 't': row.month_ce.date(), 'series': BEST[(g, h)], 'recomputed': again, 'stored_04': stored})
checks = pd.DataFrame(checks)
assert np.allclose(checks.recomputed, checks.stored_04, atol=1e-6), checks
display(checks.round(4))
print('ฟังก์ชันพยากรณ์ของ dashboard ให้ค่าเท่ากับขั้น 04 ทุก horizon')

เดือนฐาน t = 2026-07; forecast origin = 2026-09-15; ข้อมูลภายนอกเดือน t+1 ถึง 2026-08-31


,group,h,t,series,recomputed,stored_04
0,318,2,2026-06-01,ridge/3,-5.8093,-5.8093
1,318,3,2026-05-01,ridge/3,1.0964,1.0964
2,318,4,2026-04-01,ridge/3,11.2966,11.2966
3,318,5,2026-03-01,ridge/3,13.4400,13.4400
4,318,6,2026-02-01,ridge/3,9.8063,9.8063
5,402,2,2026-06-01,arima/none,-0.9099,-0.9099
6,402,3,2026-05-01,ridge/3-cutoff24,-0.0616,-0.0616
7,402,4,2026-04-01,arima/none,-1.4479,-1.4479
8,402,5,2026-03-01,arima/none,-0.4727,-0.4727
9,402,6,2026-02-01,arima/none,-2.9901,-2.9901


ฟังก์ชันพยากรณ์ของ dashboard ให้ค่าเท่ากับขั้น 04 ทุก horizon


## 2. ค่าพยากรณ์ ณ origin ล่าสุด

ค่าพยากรณ์ y คือการเปลี่ยนแปลง log สะสมจากเดือน t (×100) แปลงเป็นระดับดัชนีด้วย IMI[t] × exp(y/100)
- ช่วงความคลาดเคลื่อนคือ ± MAE ช่วง test ของโมเดลเดียวกัน
- `passes` คือผลตามเกณฑ์ "ยังเตือนได้" ของ RQ4(ก) จากขั้น 05

In [3]:
rows, coefs = [], {}
for g in GROUPS:
    imi_t = float(imi_series[g][T])
    for h in range(2, 7):
        f = forecast(g, h, T, latest_rows.loc[g], ORIGIN)
        mae = float(test_metrics[(test_metrics.commodity_code == g) & (test_metrics.horizon == h) & (test_metrics.series == f['series'])].MAE.iloc[0])
        r4 = rq4[(rq4.commodity_code == g) & (rq4.horizon == h)].iloc[0]
        rows.append({'group': g, 'h': h, 'target_month': T + pd.DateOffset(months=h), 'series': f['series'],
                     'y_pred': f['y_pred'], 'mae_test': mae, 'imi_t': imi_t,
                     'index_pred': imi_t * np.exp(f['y_pred'] / 100),
                     'index_low': imi_t * np.exp((f['y_pred'] - mae) / 100), 'index_high': imi_t * np.exp((f['y_pred'] + mae) / 100),
                     'threshold': f['threshold'], 'high_risk': f['y_pred'] > f['threshold'],
                     'skill_test': r4.skill, 'p_test': r4.p_value, 'passes': bool(r4.passes), 'train_n': f['train_n']})
        if 'coefficients' in f and h == 3: coefs[g] = f['coefficients']
forecasts = pd.DataFrame(rows)
assert np.isfinite(forecasts[['y_pred', 'index_pred', 'index_low', 'index_high']]).all().all()
forecasts.to_csv(METRICS_DIR / '06_latest_forecasts.csv', index=False)
display(forecasts.assign(target_month=forecasts.target_month.dt.strftime('%Y-%m')).round(3))

,group,h,target_month,series,y_pred,mae_test,imi_t,index_pred,index_low,index_high,threshold,high_risk,skill_test,p_test,passes,train_n
0,318,2,2026-09,ridge/3,-4.327,4.630,95.1,91.073,86.953,95.389,2.224,False,0.052,0.413,False,305
1,318,3,2026-10,ridge/3,-5.508,6.367,95.1,90.004,84.452,95.921,3.178,False,0.017,0.798,False,304
2,318,4,2026-11,ridge/3,-7.361,7.651,95.1,88.351,81.844,95.376,4.212,False,-0.005,0.954,False,303
3,318,5,2026-12,ridge/3,-5.785,8.651,95.1,89.755,82.316,97.866,5.074,False,-0.020,0.770,False,302
4,318,6,2027-01,ridge/3,-5.925,10.101,95.1,89.629,81.017,99.156,5.932,False,-0.052,0.487,False,301
5,402,2,2026-09,arima/none,-1.020,1.488,134.2,132.838,130.875,134.830,2.108,False,0.183,0.022,True,305
6,402,3,2026-10,ridge/3-cutoff24,-0.635,2.199,134.2,133.350,130.449,136.315,3.095,False,0.134,0.063,True,304
7,402,4,2026-11,arima/none,-1.777,2.689,134.2,131.837,128.339,135.429,4.413,False,0.185,0.056,True,303
8,402,5,2026-12,arima/none,-2.120,3.248,134.2,131.385,127.186,135.723,5.193,False,0.186,0.083,True,302
9,402,6,2027-01,arima/none,-2.338,3.841,134.2,131.099,126.159,136.233,6.203,False,0.168,0.146,False,301


## 3. สร้าง dashboard HTML

ออกแบบสำหรับผู้ใช้ที่ไม่ใช่นักสถิติ เช่น เจ้าของร้านปุ๋ยหรือคาเฟ่ โดยใช้หลัก 3 ข้อ
1. **ตอบคำถามหลักก่อน (answer first):** หน้าแรกบอกเป็นภาษาธรรมดาว่าราคานำเข้าเดือนเป้าหมายคาดว่าขึ้น ลง หรือทรงตัว
2. **บอกความไม่แน่นอนคู่กับตัวเลขเสมอ:** แสดงช่วงความคลาดเคลื่อนเฉลี่ยในอดีต และระดับความน่าเชื่อถือ 3 ระดับจากผลช่วง test ตามแนวทางของ van der Bles et al. (2019) ซึ่งพบว่าการบอกความไม่แน่นอนอย่างชัดเจนไม่ได้ลดความเชื่อถือของผู้อ่าน
3. **เปิดเผยรายละเอียดทีละขั้น (progressive disclosure):** ศัพท์และตารางทางเทคนิค เช่น MAE, DM-HLN และชื่อโมเดล อยู่ในส่วนที่กดเปิดได้ ตามหลักของ Nielsen Norman Group

ระดับความน่าเชื่อถือแต่ละเดือนมาจากผลขั้น 05 ของ horizon นั้น
- **พอใช้ประกอบได้:** ผ่านเกณฑ์ skill > 0 และ DM-HLN p < 0.10
- **ยังไม่แน่นอน:** skill > 0 แต่ p ≥ 0.10
- **ไม่ดีกว่าการเดาว่าราคาไม่เปลี่ยน:** skill ≤ 0

ไฟล์ HTML ฝัง plotly.js ไว้ จึงเปิดได้โดยไม่ต้องต่ออินเทอร์เน็ต

In [4]:
FONT = "'IBM Plex Sans Thai', 'Sarabun', 'Noto Sans Thai', system-ui, sans-serif"
CONFIG = {'displayModeBar': False, 'responsive': True}
GROUP_SHORT = {'318': 'ปุ๋ย', '402': 'นมและผลิตภัณฑ์นม'}
REL = {'good': ('พอใช้ประกอบได้', '#1a7f4b', '#e7f4ec'),
       'mid': ('ยังไม่แน่นอน', '#9a5b00', '#fdf1dc'),
       'low': ('ไม่ดีกว่าการเดาว่าราคาไม่เปลี่ยน', '#6b6b66', '#efeee9')}

def th_short(ts):
    ts = pd.Timestamp(ts); return f'{TH_MONTHS[ts.month - 1]} {str(ts.year + 543)[-2:]}'

def reliability(row):
    if row.passes: return 'good'
    return 'mid' if row.skill_test > 0 else 'low'

def direction(y):
    a = abs(y)
    if a < 1: return 'ทรงตัว', '↔'
    if a < 3: return ('เพิ่มขึ้นเล็กน้อย', '▲') if y > 0 else ('ลดลงเล็กน้อย', '▼')
    return ('เพิ่มขึ้น', '▲') if y > 0 else ('ลดลง', '▼')

forecasts['level'] = [reliability(r) for r in forecasts.itertuples()]

def base_layout(fig, height):
    fig.update_layout(height=height, margin=dict(l=56, r=16, t=40, b=40), paper_bgcolor='white', plot_bgcolor='white',
                      font=dict(family=FONT, size=14, color=INK), hoverlabel=dict(font_family=FONT, font_size=14),
                      legend=dict(orientation='h', yanchor='bottom', y=1.02, x=0, font=dict(size=13)))
    fig.update_xaxes(showgrid=False, linecolor=GRID, ticks='outside', tickcolor=GRID)
    fig.update_yaxes(gridcolor=GRID, zeroline=False, linecolor=GRID)
    return fig

def month_ticks(start, end):
    months = [m for m in pd.date_range(start, end, freq='MS') if m.month in (1, 7)]
    return dict(tickvals=months, ticktext=[th_short(m) for m in months])

def forecast_chart(g):
    hist = imi_series[g][T - pd.DateOffset(months=35):T]
    fc = forecasts[forecasts.group == g]
    color = GROUP_COLOR[g]
    fill = 'rgba(42,120,214,0.13)' if g == '318' else 'rgba(27,175,122,0.15)'
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=hist.index, y=hist.values, mode='lines', name='ราคานำเข้าจริง', line=dict(color=INK, width=2.5),
                             customdata=[th_month(m) for m in hist.index],
                             hovertemplate='%{customdata}<br>ดัชนี %{y:.1f}<extra></extra>'))
    fig.add_trace(go.Scatter(x=[T, *fc.target_month, *fc.target_month[::-1], T],
                             y=[hist.iloc[-1], *fc.index_high, *fc.index_low[::-1], hist.iloc[-1]],
                             mode='lines', fill='toself', fillcolor=fill, line=dict(width=0), hoverinfo='skip',
                             name='ช่วงความคลาดเคลื่อนเฉลี่ยในอดีต'))
    custom = [[th_month(T), 'ข้อมูลจริงล่าสุด', 0.0]] + [
        [th_month(r.target_month), REL[r.level][0], r.y_pred] for r in fc.itertuples()]
    fig.add_trace(go.Scatter(x=[T, *fc.target_month], y=[hist.iloc[-1], *fc.index_pred], mode='lines+markers',
                             name='ค่าคาดการณ์', line=dict(color=color, width=2.5, dash='dot'),
                             marker=dict(size=12, color=['white'] + [REL[l][1] for l in fc.level], line=dict(color=color, width=2)),
                             customdata=custom,
                             hovertemplate='%{customdata[0]}<br>ดัชนีคาด %{y:.1f} (%{customdata[2]:+.1f}%)<br>ความน่าเชื่อถือ: %{customdata[1]}<extra></extra>'))
    fig.add_vline(x=T, line=dict(color=MUTED, width=1, dash='dot'))
    fig.add_annotation(x=T, y=1.0, yref='paper', text='ข้อมูลจริงล่าสุด', showarrow=False, xanchor='right', yanchor='bottom',
                       font=dict(size=12, color=MUTED))
    fig.update_xaxes(**month_ticks(hist.index.min(), fc.target_month.max()))
    fig.update_yaxes(title_text='ดัชนี (ปี 2555 = 100)')
    return base_layout(fig, 360)

def coef_chart(g):
    if g not in coefs: return None
    c = coefs[g].reindex(coefs[g].abs().sort_values(ascending=True).index).tail(8)
    fig = go.Figure(go.Bar(x=c.values, y=c.index, orientation='h', marker=dict(color=GROUP_COLOR[g]),
                           hovertemplate='%{y}<br>coefficient %{x:+.3f}<extra></extra>'))
    fig.add_vline(x=0, line=dict(color=MUTED, width=1))
    fig.update_xaxes(title_text='coefficient (feature ปรับมาตรฐานแล้ว)', title_font=dict(size=12))
    fig.update_layout(showlegend=False)
    return base_layout(fig, 300)

def drivers_chart():
    ext = external[(external.month_ce >= T - pd.DateOffset(months=35)) & (external.month_ce <= T + pd.DateOffset(months=1))]
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.16,
                        subplot_titles=('อัตราแลกเปลี่ยน (บาทต่อดอลลาร์)', 'ราคาน้ำมัน Brent (ดอลลาร์ต่อบาร์เรล)'))
    for i, (sid, unit, color) in enumerate((('DEXTHUS', 'บาท/ดอลลาร์', INK), ('DCOILBRENTEU', 'ดอลลาร์/บาร์เรล', '#eb6834')), start=1):
        s = ext[ext.series == sid]
        fig.add_trace(go.Scatter(x=s.month_ce, y=s.full_mean, mode='lines', line=dict(color=color, width=2), name=sid,
                                 customdata=[th_month(m) for m in s.month_ce],
                                 hovertemplate=f'%{{customdata}}<br>%{{y:.2f}} {unit}<extra></extra>'), row=i, col=1)
    fig.update_xaxes(**month_ticks(ext.month_ce.min(), ext.month_ce.max()))
    fig.update_annotations(font=dict(size=13, color=MUTED), x=0, xanchor='left')
    fig.update_layout(showlegend=False)
    return base_layout(fig, 380)

figs = {g: {'forecast': forecast_chart(g), 'coef': coef_chart(g)} for g in GROUPS}
drivers_html = drivers_chart().to_html(full_html=False, include_plotlyjs=False, config=CONFIG)
display(forecasts[['group', 'h', 'target_month', 'y_pred', 'skill_test', 'p_test', 'level']].assign(
    target_month=lambda d: d.target_month.dt.strftime('%Y-%m')).round(3))

,group,h,target_month,y_pred,skill_test,p_test,level
0,318,2,2026-09,-4.327,0.052,0.413,mid
1,318,3,2026-10,-5.508,0.017,0.798,mid
2,318,4,2026-11,-7.361,-0.005,0.954,low
3,318,5,2026-12,-5.785,-0.020,0.770,low
4,318,6,2027-01,-5.925,-0.052,0.487,low
5,402,2,2026-09,-1.020,0.183,0.022,good
6,402,3,2026-10,-0.635,0.134,0.063,good
7,402,4,2026-11,-1.777,0.185,0.056,good
8,402,5,2026-12,-2.120,0.186,0.083,good
9,402,6,2027-01,-2.338,0.168,0.146,mid


In [5]:
def esc(x): return html.escape(str(x))
def pval(p): return '< 0.001' if p < 0.001 else f'{p:.3f}'

def badge(level, extra=''):
    label, fg, bg = REL[level]
    return f'<span class="badge" style="color:{fg};background:{bg}">● {label}</span>{extra}'

def reliability_reason(r):
    s = 100 * r.skill_test
    if r.level == 'good':
        return f'ในช่วงทดสอบปี 2562–2569 ค่าคาดการณ์คลาดเคลื่อนน้อยกว่าการเดาว่าราคาไม่เปลี่ยน {s:.0f}%'
    if r.level == 'mid':
        return f'ในช่วงทดสอบ แม่นกว่าการเดาว่าราคาไม่เปลี่ยนเพียง {s:.0f}% ซึ่งยังแยกจากความบังเอิญไม่ได้'
    return f'ในช่วงทดสอบ คลาดเคลื่อนมากกว่าการเดาว่าราคาไม่เปลี่ยน {abs(s):.0f}%'

def hero(g):
    r = forecasts[(forecasts.group == g) & (forecasts.h == 3)].iloc[0]
    word, arrow = direction(r.y_pred)
    low, high = r.y_pred - r.mae_test, r.y_pred + r.mae_test
    crosses = low < 0 < high
    return f"""
    <div class="hero">
      <div class="hero-q">ราคานำเข้า{GROUP_SHORT[g]} เดือน {th_month(r.target_month)} คาดว่า</div>
      <div class="hero-a" style="color:{GROUP_COLOR[g]}">{arrow} {word}</div>
      <div class="hero-n">ประมาณ <b>{r.y_pred:+.1f}%</b> เมื่อเทียบกับ {th_month(T)}
        <span class="muted">· ในอดีตคลาดเคลื่อนเฉลี่ย ±{r.mae_test:.1f} จุดร้อยละ จึงอาจอยู่ระหว่าง {low:+.1f}% ถึง {high:+.1f}%</span></div>
      {'<div class="hero-note">ช่วงนี้ครอบทั้งขึ้นและลง จึงยังบอกทิศทางได้ไม่แน่ชัด</div>' if crosses else ''}
      <div class="hero-rel">ความน่าเชื่อถือ: {badge(r.level)} <span class="muted">{reliability_reason(r)}</span></div>
    </div>"""

def month_strip(g):
    cards = []
    for r in forecasts[forecasts.group == g].itertuples():
        word, arrow = direction(r.y_pred)
        label, fg, bg = REL[r.level]
        when = 'เดือนที่ประเมิน' if r.h == 2 else f'อีก {r.h - 2} เดือน'
        cards.append(f"""<div class="mcard" style="border-top:4px solid {fg}">
          <div class="m-month">{th_month(r.target_month)}</div><div class="m-when">{when}</div>
          <div class="m-dir">{arrow} {word}</div><div class="m-val">{r.y_pred:+.1f}%</div>
          <div class="m-rel" style="color:{fg}">● {label}</div></div>""")
    return f'<div class="strip">{"".join(cards)}</div>'

USE = {
    '402': ['ใช้เป็นสัญญาณเริ่มต้นก่อนคุยราคากับ supplier หรือวางงบซื้อวัตถุดิบนมนำเข้า',
            'ข้อมูลย้อนหลังชี้ว่าราคาขายปลีกนมในประเทศมักเปลี่ยนตามราคานำเข้าช้ากว่าประมาณ 4 เดือน และเปลี่ยนเพียงบางส่วน (ยังไม่ได้ยืนยันว่าช่วยเตือนได้จริง)',
            'ความแม่นมาจากแนวโน้มของราคานำเข้าเอง ไม่ได้มาจากค่าเงินหรือราคาน้ำมัน'],
    '318': ['ตัวเลขของปุ๋ยยังไม่น่าเชื่อถือพอ ใช้ดูประกอบเท่านั้น อย่าใช้ตัดสินใจสต็อกเพียงลำพัง',
            'ค่าเงินและราคาน้ำมันช่วยพยากรณ์ราคาปุ๋ยนำเข้าได้น้อย ควรติดตามราคาปุ๋ยโลก (ยูเรีย, DAP) ประกอบ',
            'การทดลองเพิ่มเติมพบว่าราคาปุ๋ยโลกมีแนวโน้มช่วยให้แม่นขึ้น แต่ยังต้องยืนยันกับข้อมูลใหม่'],
}
NOT_TELL = ['ไม่ใช่ราคาที่ร้านของคุณจะจ่ายจริง เป็นดัชนีราคานำเข้าระดับกลุ่มสินค้า คิดเป็นดอลลาร์สหรัฐ',
            'ไม่ได้บอกให้ซื้อหรือไม่ซื้อ และไม่ใช่คำแนะนำทางการเงิน',
            'คาดการณ์เหตุการณ์ที่ไม่เคยเกิดในอดีตไม่ได้ เช่น สงครามหรือมาตรการส่งออกใหม่']

def eval_table(g):
    fc = forecasts[forecasts.group == g]
    i2 = iter2[iter2.commodity_code == g].set_index('horizon')
    body = ''.join(
        f"<tr><td>h={r.h} · {th_month(r.target_month)}</td><td>{esc(r.series)}</td><td class='num'>{r.y_pred:+.2f}%</td>"
        f"<td class='num'>±{r.mae_test:.2f}</td><td class='num'>{100 * r.skill_test:+.1f}%</td><td class='num'>{pval(r.p_test)}</td>"
        f"<td>{REL[r.level][0]}</td><td class='num muted'>{100 * i2.loc[r.h, 'skill_vs_no_change']:+.1f}% (p {pval(i2.loc[r.h, 'p_vs_no_change'])})</td></tr>"
        for r in fc.itertuples())
    return f"""<div class="tablewrap"><table><thead><tr><th>horizon · เดือน</th><th>โมเดล</th><th class='num'>ค่าคาด y</th>
      <th class='num'>MAE ช่วง test</th><th class='num'>skill เทียบ No-change</th><th class='num'>DM-HLN p</th><th>ระดับ</th>
      <th class='num'>รอบที่ 2 (เชิงสำรวจ)</th></tr></thead><tbody>{body}</tbody></table></div>"""

def technical(g):
    r = forecasts[(forecasts.group == g) & (forecasts.h == 3)].iloc[0]
    hr = alerts[(alerts.commodity_code == g) & (alerts.series == r.series)].iloc[0]
    always = alerts[(alerts.commodity_code == g) & (alerts.series == 'always_alert')].iloc[0]
    c1 = dm_main[(dm_main.commodity_code == g) & dm_main.comparison.str.startswith('C1')].iloc[0]
    prec = 'ไม่กำหนด (ไม่มีการเตือน)' if pd.isna(hr.precision) else f'{hr.precision:.2f}'
    coef_html = (figs[g]['coef'].to_html(full_html=False, include_plotlyjs=False, config=CONFIG) if figs[g]['coef'] is not None
                 else '<p class="muted">โมเดลที่เลือกสำหรับ h=3 ไม่ใช่ Ridge จึงไม่มี coefficient</p>')
    return f"""
    <details class="tech"><summary>รายละเอียดทางเทคนิค (สำหรับผู้สนใจวิธีการ)</summary>
      <h4>ผลการประเมินช่วง test (เดือนฐาน ม.ค. 2562 เป็นต้นไป)</h4>
      <p class="muted">h นับจากเดือนฐาน t; forecast ออกวันที่ 15 ของ t+2 ดังนั้น h=2 คือเดือนที่ออก forecast · ±MAE ไม่ใช่ช่วงความเชื่อมั่น</p>
      {eval_table(g)}
      <h4>ข้อมูลภายนอกช่วยหรือไม่ (h=3)</h4>
      <p>เพิ่มอัตราแลกเปลี่ยนและราคาน้ำมัน: MAE {c1.MAE_1:.2f} เทียบใช้ประวัติราคาอย่างเดียว {c1.MAE_2:.2f} (Holm p = {c1.p_holm:.2f}) ไม่มีนัยสำคัญ</p>
      <h4>ป้ายเตือน High Risk (ไม่แสดงในหน้าหลัก)</h4>
      <p>สถานะปัจจุบัน: {'High Risk' if r.high_risk else 'Normal'} (threshold {r.threshold:.1f}%) · ช่วง test precision {prec}, recall {hr.recall:.2f}, F1 {hr.F1:.2f}
      เทียบกับการเตือนทุกเดือน F1 {always.F1:.2f} สัญญาณนี้จึงยังไม่ดีกว่าการเตือนทุกเดือน</p>
      <h4>ปัจจัยที่โมเดล h=3 ให้น้ำหนัก ({esc(r.series)})</h4>
      {coef_html}
    </details>"""

def section(g, active):
    use = ''.join(f'<li>{t}</li>' for t in USE[g])
    nott = ''.join(f'<li>{t}</li>' for t in NOT_TELL)
    return f"""
  <section id="g{g}" class="panel" {'hidden' if not active else ''}>
    {hero(g)}
    <h2>คาดการณ์รายเดือน</h2>
    {month_strip(g)}
    <div class="legend">ความน่าเชื่อถือวัดจากผลทดสอบย้อนหลังปี 2562–2569 เทียบกับการเดาว่าราคาไม่เปลี่ยน:
      {badge('good')} {badge('mid')} {badge('low')}</div>
    <div class="card"><h3>ราคานำเข้าย้อนหลัง 3 ปี และค่าคาดการณ์</h3>
      <p class="muted">เส้นทึบคือราคาจริง เส้นประคือค่าคาดการณ์ แถบสีคือช่วงความคลาดเคลื่อนเฉลี่ยในอดีต สีของจุดบอกระดับความน่าเชื่อถือ</p>
      {figs[g]['forecast'].to_html(full_html=False, include_plotlyjs=False, config=CONFIG)}</div>
    <div class="grid2">
      <div class="card"><h3>ใช้ข้อมูลนี้อย่างไร</h3><ul>{use}</ul></div>
      <div class="card"><h3>สิ่งที่ข้อมูลนี้ไม่ได้บอก</h3><ul>{nott}</ul></div>
    </div>
    {technical(g)}
  </section>"""

GLOSSARY = [
    ('ดัชนีราคานำเข้า (IMI)', 'ดัชนีราคาสินค้านำเข้าของไทยรายกลุ่มสินค้าจาก สนค. คิดเป็นดอลลาร์สหรัฐ ปี 2555 = 100'),
    ('การเดาว่าราคาไม่เปลี่ยน (No-change)', 'ตัวเทียบพื้นฐาน ทายว่าราคาเดือนเป้าหมายเท่ากับเดือนล่าสุดที่รู้'),
    ('ความคลาดเคลื่อนเฉลี่ย (MAE)', 'ค่าเฉลี่ยของความต่างระหว่างค่าคาดการณ์กับค่าจริงในช่วงทดสอบ หน่วยเป็นจุดร้อยละ'),
    ('ช่วงทดสอบ', 'เดือนฐาน ม.ค. 2562 เป็นต้นไป โดยทุกเดือนใช้เฉพาะข้อมูลที่เผยแพร่แล้ว ณ วันที่ประเมิน'),
    ('วันที่ประเมิน', 'วันที่ 15 ของเดือนที่สองหลังเดือนข้อมูลล่าสุด ซึ่งเป็นวันที่ข้อมูลทุกแหล่งเผยแพร่แล้ว'),
]
glossary_html = ''.join(f'<dt>{a}</dt><dd>{b}</dd>' for a, b in GLOSSARY)

page = f"""<!doctype html>
<html lang="th"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>แนวโน้มราคานำเข้า ปุ๋ยและนม</title>
<link rel="preconnect" href="https://fonts.googleapis.com"><link href="https://fonts.googleapis.com/css2?family=IBM+Plex+Sans+Thai:wght@400;600&display=swap" rel="stylesheet">
<style>
:root {{ --bg:#f7f6f2; --surface:#ffffff; --ink:{INK}; --muted:{MUTED}; --line:{GRID}; }}
* {{ box-sizing:border-box; }}
body {{ margin:0; background:var(--bg); color:var(--ink); font-family:{FONT}; font-size:17px; line-height:1.6; }}
.wrap {{ max-width:1080px; margin:0 auto; padding:24px 16px 48px; }}
header h1 {{ margin:0 0 4px; font-size:26px; }} header p {{ margin:0; color:var(--muted); font-size:15px; }}
.notice {{ margin:14px 0 8px; padding:10px 14px; background:#fff7e6; border:1px solid #f0d9a8; border-radius:8px; font-size:15px; }}
nav {{ display:flex; gap:8px; margin:16px 0; flex-wrap:wrap; }}
nav button {{ font:inherit; font-size:17px; padding:10px 18px; border-radius:999px; border:1px solid var(--line); background:var(--surface); cursor:pointer; }}
nav button[aria-selected="true"] {{ background:var(--ink); color:#fff; border-color:var(--ink); }}
.hero {{ background:var(--surface); border:1px solid var(--line); border-radius:12px; padding:20px 22px; margin-bottom:16px; }}
.hero-q {{ font-size:18px; }} .hero-a {{ font-size:40px; font-weight:600; line-height:1.25; margin:4px 0 6px; }}
.hero-n {{ font-size:17px; }} .hero-note {{ margin-top:6px; color:#9a5b00; }}
.hero-rel {{ margin-top:10px; font-size:15px; }}
.badge {{ display:inline-block; padding:2px 10px; border-radius:999px; font-size:14px; font-weight:600; margin-right:6px; }}
h2 {{ font-size:19px; margin:18px 0 8px; }}
.strip {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(150px,1fr)); gap:10px; }}
.mcard {{ background:var(--surface); border:1px solid var(--line); border-radius:10px; padding:10px 12px; }}
.m-month {{ font-weight:600; }} .m-when {{ font-size:13px; color:var(--muted); }}
.m-dir {{ margin-top:6px; }} .m-val {{ font-size:22px; font-weight:600; }} .m-rel {{ font-size:13px; font-weight:600; }}
.legend {{ margin:10px 0 16px; font-size:14px; color:var(--muted); }}
.card {{ background:var(--surface); border:1px solid var(--line); border-radius:12px; padding:14px 16px; margin-bottom:14px; }}
.card h3 {{ margin:0 0 4px; font-size:18px; }} .card ul {{ margin:6px 0 0; padding-left:20px; }} .card li {{ margin-bottom:6px; }}
.grid2 {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(300px,1fr)); gap:14px; }}
.muted {{ color:var(--muted); font-weight:400; }}
details.tech, details.more {{ background:var(--surface); border:1px solid var(--line); border-radius:12px; padding:12px 16px; margin-bottom:14px; }}
details summary {{ cursor:pointer; font-weight:600; font-size:16px; }}
details h4 {{ margin:14px 0 4px; font-size:16px; }} details p {{ font-size:15px; margin:4px 0; }}
.tablewrap {{ overflow-x:auto; }}
table {{ border-collapse:collapse; width:100%; font-size:14px; }} th, td {{ padding:6px 8px; border-bottom:1px solid var(--line); text-align:left; white-space:nowrap; }}
th {{ font-weight:600; }} .num {{ text-align:right; font-variant-numeric:tabular-nums; }}
dl {{ margin:6px 0 0; }} dt {{ font-weight:600; margin-top:8px; }} dd {{ margin:0 0 0 0; color:var(--muted); font-size:15px; }}
footer {{ margin-top:20px; color:var(--muted); font-size:13px; }}
</style>
<script>{plotly.offline.get_plotlyjs()}</script>
</head><body><div class="wrap">
<header><h1>แนวโน้มราคานำเข้าปุ๋ยและผลิตภัณฑ์นม</h1>
<p>ประเมิน ณ {th_date(ORIGIN)} จากราคานำเข้าจริงถึง {th_month(T)} · อัปเดตครั้งถัดไป {th_date(ORIGIN + pd.DateOffset(months=1))}</p></header>
<div class="notice">ใช้ประกอบการวางแผนเท่านั้น ตัวเลขเป็นดัชนีราคานำเข้าระดับกลุ่มสินค้า ไม่ใช่ราคาที่ร้านของคุณจะจ่าย และไม่ใช่คำแนะนำให้ซื้อหรือไม่ซื้อ</div>
<nav role="tablist">{''.join(f'<button role="tab" data-target="g{g}" aria-selected="{str(i == 0).lower()}">{esc(GROUP_TH[g])}</button>' for i, g in enumerate(GROUPS))}</nav>
{''.join(section(g, i == 0) for i, g in enumerate(GROUPS))}
<details class="more"><summary>ปัจจัยภายนอก: อัตราแลกเปลี่ยนและราคาน้ำมัน 3 ปีล่าสุด</summary>
  <p class="muted">ผลการทดสอบพบว่าปัจจัยทั้งสองไม่ได้ช่วยพยากรณ์ราคานำเข้าปุ๋ยและนมอย่างมีนัยสำคัญ แสดงไว้เพื่อติดตามสถานการณ์</p>
  {drivers_html}</details>
<details class="more"><summary>คำศัพท์และวิธีอ่าน</summary><dl>{glossary_html}</dl></details>
<footer>แหล่งข้อมูล: สนค. กระทรวงพาณิชย์ (IMI, CPI, PPI; Open Data Common) · FRED: DEXTHUS (Federal Reserve H.10), DCOILBRENTEU (U.S. EIA) · World Bank Pink Sheet (CC BY 4.0) · FAO Food Price Index ·
วิธีการและข้อจำกัดทั้งหมดอยู่ใน PROJECT_SCOPE.md และ notebook 01–06 · สร้างเมื่อ {datetime.now():%Y-%m-%d %H:%M}</footer>
</div>
<script>
function resizeIn(el) {{ el.querySelectorAll('.plotly-graph-div').forEach(d => Plotly.Plots.resize(d)); }}
function showTab(id) {{
  document.querySelectorAll('nav button').forEach(b => b.setAttribute('aria-selected', b.dataset.target === id));
  document.querySelectorAll('.panel').forEach(p => p.hidden = p.id !== id);
  resizeIn(document.getElementById(id));
}}
document.querySelectorAll('nav button').forEach(btn => btn.addEventListener('click', () => {{
  showTab(btn.dataset.target); history.replaceState(null, '', '#' + btn.dataset.target);
}}));
document.querySelectorAll('details').forEach(d => d.addEventListener('toggle', () => {{ if (d.open) resizeIn(d); }}));
if (document.getElementById(location.hash.slice(1))) showTab(location.hash.slice(1));
</script></body></html>"""
out_path = DASH_DIR / 'imi_dashboard.html'
out_path.write_text(page, encoding='utf-8')
size_mb = out_path.stat().st_size / 1e6
assert all(f'id="g{g}"' in page for g in GROUPS) and size_mb < 16
print(f'บันทึก {out_path.relative_to(ROOT)} ({size_mb:.1f} MB)')

บันทึก reports/dashboard/imi_dashboard.html (4.9 MB)


## 4. Checkpoint

เงื่อนไขผ่านตาม Scope หัวข้อ 15.3: แสดงผลได้ถูกต้องกับข้อมูลเดือนล่าสุด
- origin ไม่เกินวันที่ as-of
- ข้อมูลภายนอกเดือน t+1 ครบก่อน origin
- ฟังก์ชันพยากรณ์ให้ค่าเท่ากับขั้น 04
- ค่าพยากรณ์ครบทุกกลุ่มและ horizon
- ไฟล์ HTML มีทั้งสองกลุ่ม

In [6]:
assert ORIGIN <= AS_OF
assert len(forecasts) == len(GROUPS) * 5
assert (forecasts.train_n > 0).all()
print('CHECKPOINT PASSED: dashboard ใช้ข้อมูลถึงเดือนฐาน', f'{T:%Y-%m}', 'และค่าพยากรณ์ตรวจสอบกับขั้น 04 แล้ว')

log_path = ROOT / 'logs/process_log.md'
log = log_path.read_text(encoding='utf-8')
marker = 'ขั้นตอน: 06 Deployment'
if marker not in log:
    fc3 = forecasts[forecasts.h == 3].set_index('group')
    entry = f"""

## [2026-10-05] {marker}
**สิ่งที่ทำ:** สร้าง dashboard HTML 1 หน้า (reports/dashboard/imi_dashboard.html) ที่ฝัง plotly.js ไว้ในไฟล์ แสดงค่าพยากรณ์ h=2–6 ของกลุ่มปุ๋ยและนม ณ origin ล่าสุด พร้อมผลการประเมินช่วง test ผลรอบที่ 2 (ติดป้ายเชิงสำรวจ) และข้อมูลภายนอก

**ผลที่ได้:**
- as-of {AS_OF.date()} เดือนฐาน t = {T:%Y-%m} forecast origin {ORIGIN.date()}
- ค่าพยากรณ์ h=3: ปุ๋ย {fc3.loc['318', 'y_pred']:+.2f}% ({fc3.loc['318', 'series']}); นม {fc3.loc['402', 'y_pred']:+.2f}% ({fc3.loc['402', 'series']})
- ฟังก์ชันพยากรณ์ของ dashboard ให้ค่าเท่ากับขั้น 04 ทุก horizon เมื่อคำนวณซ้ำที่ origin สุดท้ายของช่วง test
- ไฟล์ขนาด {size_mb:.1f} MB และบันทึกค่าพยากรณ์ไว้ที่ logs/metrics/06_latest_forecasts.csv

**สิ่งที่พบ / ปัญหา:** IMI เดือน {T + pd.DateOffset(months=1):%Y-%m} มีใน API แล้ว แต่ตามกฎ forecast origin จะใช้ได้ตั้งแต่วันที่ {(ORIGIN + pd.DateOffset(months=1)).date()} dashboard จึงยังไม่ใช้เดือนนั้น

**การตัดสินใจและเหตุผล:** แสดงคำเตือนว่าสัญญาณ High Risk ในช่วง test ไม่ได้ดีกว่าการเตือนทุกเดือน และแสดง skill กับ p-value คู่กับค่าทำนายทุก horizon เพื่อไม่ให้ผู้ใช้เชื่อค่าทำนายเกินหลักฐาน

**ขั้นต่อไป:** เขียนรายงาน CRISP-DM จาก process log และเตรียมนำเสนอ
"""
    log_path.write_text(log.rstrip() + entry + '\n', encoding='utf-8')
    print('บันทึกผลลง logs/process_log.md แล้ว')
else:
    print('มีรายการ 06 Deployment ใน process log แล้ว จึงไม่บันทึกซ้ำ')

CHECKPOINT PASSED: dashboard ใช้ข้อมูลถึงเดือนฐาน 2026-07 และค่าพยากรณ์ตรวจสอบกับขั้น 04 แล้ว
มีรายการ 06 Deployment ใน process log แล้ว จึงไม่บันทึกซ้ำ


In [7]:
marker_r = 'ขั้นตอน: 06 Deployment (ปรับหน้า dashboard ให้อ่านง่ายสำหรับผู้ใช้ทั่วไป)'
log = log_path.read_text(encoding='utf-8')
if marker_r not in log:
    counts = forecasts.level.value_counts().to_dict()
    entry = f"""

## [2026-10-05] {marker_r}
**สิ่งที่ทำ:** ออกแบบหน้า dashboard ใหม่ตามหลัก answer first, การสื่อสารความไม่แน่นอนคู่กับตัวเลข (van der Bles et al., 2019) และ progressive disclosure (Nielsen Norman Group)
- หน้าหลักบอกทิศทางราคาเดือนเป้าหมายเป็นภาษาธรรมดา พร้อมช่วงความคลาดเคลื่อนเฉลี่ยในอดีตและระดับความน่าเชื่อถือ 3 ระดับ
- ย้ายศัพท์และตารางทางเทคนิค ป้าย High Risk และ coefficient ไปไว้ในส่วนที่กดเปิดได้

**ผลที่ได้:** ระดับความน่าเชื่อถือของค่าพยากรณ์ 10 ค่า: {counts} ไฟล์ขนาด {size_mb:.1f} MB ค่าพยากรณ์และตัวเลขการประเมินไม่เปลี่ยน

**สิ่งที่พบ / ปัญหา:** หน้าเดิมใช้ศัพท์เทคนิค (h, MAE, DM-HLN, skill, ชื่อโมเดล) ในหน้าหลัก และแสดงป้าย High Risk ที่ผลทดสอบชี้ว่ายังใช้ไม่ได้ไว้ด้านบน ผู้ใช้ทั่วไปจึงอ่านยาก
- scripts/review_dashboard.py ของการตรวจทานอ้างโครงสร้างหน้าเดิม ต้องปรับก่อนใช้ตรวจซ้ำ

**การตัดสินใจและเหตุผล:** ระดับความน่าเชื่อถือใช้เกณฑ์เดิมจากขั้น 05 (skill > 0 และ p < 0.10) ไม่ได้สร้างเกณฑ์ใหม่ และคำว่า ทรงตัว/เล็กน้อย ใช้เกณฑ์ |y| < 1% และ < 3% เพื่อการสื่อสารเท่านั้น

**ขั้นต่อไป:** รอทีมตรวจหน้าตา dashboard
"""
    log_path.write_text(log.rstrip() + entry + '\n', encoding='utf-8')
    print('บันทึกการปรับหน้า dashboard ลง logs/process_log.md แล้ว')